# Disambiguation parameter optimization

> **Notebook convention:** as in [`entity_linking.ipynb`](entity_linking.ipynb), keep this notebook
> limited to function calls and variable definitions (experiment parameters, calls whose results are
> displayed). Every function, class and significant chunk of logic belongs in the support script
> [`disambiguation_optimization_utils.py`](disambiguation_optimization_utils.py), taking its
> dependencies as explicit arguments rather than reading notebook globals.

Explores the parameters of the disambiguation procedure ([`modules/geo_disambiguation.py`](modules/geo_disambiguation.py)),
scored on the **training split only** (`card_id` starting with `train_`).

**Objective**, in order of priority:
1. higher precision;
2. more correctly linked addresses;
3. fewer incorrectly linked addresses (wrong links that are not an ancestor of the true location).

**Method.** The GeoDB search does not depend on the disambiguator, so the training addresses are linked once
by the production pipeline, keeping the addresses the search returned (cached under `cache/`). Every
configuration then only replays the disambiguation of those addresses, in parallel worker processes.

**Stages.** Each stage starts from the best configuration of the previous one:
1. the weights of the weighted score;
2. the parent/child distance thresholds of the hierarchical likelihood;
3. the minimum population for the population order of magnitude;
4. the order of the disambiguation factors and their significance thresholds, by a greedy search;
5. the significance thresholds, keeping the production order (from the best of stage 3);
6. the order, by moving each factor of the production order up (from the best of stage 5).

`%autoreload` is enabled below, but the evaluation workers load the support script when they start:
rerun the cell creating the `evaluator` after editing it.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import entity_linking_notebook_utils as utils
utils.setup_logging()
import disambiguation_optimization_utils as opt
import modules.entity_linking as entity_linking

In [2]:
SEARCH_INDEX_DIR = ".geo_db_search_index"
PARSED_ADDRESSES_CACHE_PATH = Path("experiments_data/entity_linking/parsed_addresses.jsonl")
# The training addresses as the GeoDB search returned them; rebuild after changing the search or the parsing
SEARCHED_FIELDS_CACHE_PATH = Path("cache/disambiguation_optimization/searched_train_fields.pkl")
OVERWRITE_SEARCH_CACHE = False
NUM_SEARCH_WORKERS = utils.default_num_workers()
# Each evaluation worker holds every searched training address (a few hundred MB)
NUM_EVALUATION_WORKERS = 32
# Addresses per evaluation task: small enough for a single configuration to keep every worker busy
EVALUATION_CHUNK_SIZE = 25

In [3]:
ground_truth = utils.load_ground_truth("open_data/bzkopen_linking_groundtruth.jsonl")
parsed_addresses = await utils.load_or_build_parsed_addresses(ground_truth, PARSED_ADDRESSES_CACHE_PATH)
train_rows, train_ground_truth = opt.train_split(parsed_addresses, ground_truth)
print(f"Training addresses: {len(train_rows)} of {len(ground_truth)}")

Running regex parsing on 1090 addresses...
606/1090 addresses need the LLM fallback (regex left them un/partially parsed).
Retrieving cached parsed addresses from experiments_data/entity_linking/parsed_addresses.jsonl...


Training addresses: 771 of 1090


In [4]:
searched_fields = opt.load_or_search_fields(
    train_rows, SEARCHED_FIELDS_CACHE_PATH, f"{SEARCH_INDEX_DIR}/tantivy_index", NUM_SEARCH_WORKERS,
    overwrite=OVERWRITE_SEARCH_CACHE,
)

Retrieving cached searched addresses from cache/disambiguation_optimization/searched_train_fields.pkl...


## Production configuration

The configuration `entity_linking.build_disambiguator()` builds. Replaying its disambiguation must link every
training address as the pipeline did when searching.

In [5]:
opt.cleanup_previous_run(globals()) # Allow script rerunning
production_config = opt.DisambiguationConfig.of(entity_linking.build_disambiguator())
evaluator = opt.ConfigEvaluator(
    SEARCHED_FIELDS_CACHE_PATH, train_ground_truth, NUM_EVALUATION_WORKERS, EVALUATION_CHUNK_SIZE)
opt.check_replay(evaluator, production_config, searched_fields)
evaluator.evaluate({"production": production_config})

2026-10-08 14:05:07.386 INFO     entity_linking.Disambiguator geo_disambiguation.py:390 Initialized with significance thresholds defaultdict(<function _default_significance_threshold at 0x7f65c3243880>, {'fuzzy_similarity_score': 0, 'population_order_of_magnitude': 1, 'weighted_score': 0.05, 'child_parent_likelihood': 0.05, 'entity_types_matching_preferred': 0.05, 'country_likelihood_rank': 0.05, 'phonetic_score': 0.05, 'entity_types_matching': 0.05, 'is_preferred_name': 0.05, 'population_count': 0.05}), primary priority ['weighted_score', 'child_parent_likelihood', 'population_order_of_magnitude', 'entity_types_matching_preferred', 'fuzzy_similarity_score', 'country_likelihood_rank', 'phonetic_score', 'entity_types_matching', 'is_preferred_name', 'population_count'], secondary priority [], population rounding factor 1, min population order of magnitude 100000, score prune thresholds {'fuzzy_similarity_score': 0.4, 'child_parent_likelihood': 0.4}, geo db geo.duckdb, geometry parent/chi

Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

Replaying the disambiguation links all 771 addresses as the pipeline did


,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
production,0.883845,662,75,12,5,0.935028,0.992504


In [6]:
print(opt.describe_config(production_config))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'population_order_of_magnitude',
    'entity_types_matching_preferred',
    'fuzzy_similarity_score',
    'country_likelihood_rank',
    'phonetic_score',
    'entity_types_matching',
    'is_preferred_name',
    'population_count',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'phonetic_score': 2,
    'fuzzy_similarity_score': 3,
    'entity_types_matching_preferred': 2,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 100,
    1: 100,
    2: 34,
    3: 14,
    4: 6,
    5: 5,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 100_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'population_order_of_magnitude': 1,
    'entity_types_matching_preferred': 0.05,
    'fuzzy_similarity_score

In [7]:
base_congig = production_config.replace(
    primary_priority = production_config.primary_priority + production_config.secondary_priority,
    secondary_priority = tuple()
)
evaluator.evaluate({"base": production_config})

,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
base,0.883845,662,75,12,5,0.935028,0.992504


## Stage 1: weights of the weighted score

The same four factors as the production weights. Round 1 sets every weight to 1 and tries a weight of 2 for each
factor in turn. Round 2 gives the factor that did best at 2 a weight of 3, and tries the other factors at 2 except
one, which weighs 1 or is excluded in turn; plus the other factors all at 2, and all at 1. The production weights
and all weights at 1 are evaluated as well. The best configuration of round 2 is the best overall, since round 2
includes the best of round 1.

In [8]:
weights_round_1, weights_round_2 = opt.optimize_score_weights(evaluator, base_congig)
weights_round_1.evaluations

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Best factor weighing 2 in round 1: fuzzy_similarity_score


,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,2,1,0.883845,662,74,13,5,0.935028,0.992504
current,2,3,2,0.883845,662,75,12,5,0.935028,0.992504
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,0.882510,661,76,12,5,0.934276,0.992492
"phonetic_score=2, fuzzy_similarity_score=1, entity_types_matching_preferred=1",2,1,1,0.878505,658,79,12,5,0.932011,0.992459
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=2",1,1,2,0.875834,656,81,12,5,0.930496,0.992436


In [9]:
weights_round_2.evaluations

,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=1",2,3,1,0.885180,663,74,12,5,0.935780,0.992515
"phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,2,1,0.883845,662,74,13,5,0.935028,0.992504
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=2",2,3,2,0.883845,662,75,12,5,0.935028,0.992504
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,0.882510,661,76,12,5,0.934276,0.992492
"phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=1",1,3,1,0.881175,660,77,12,5,0.933522,0.992481
"phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=2",1,3,2,0.879840,659,77,13,5,0.932767,0.992470
"phonetic_score=0, fuzzy_similarity_score=3, entity_types_matching_preferred=2",0,3,2,0.878505,658,78,13,5,0.932011,0.992459
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=0",2,3,0,0.866489,649,81,19,5,0.925160,0.992355


In [10]:
after_weights = weights_round_2.best_config
print(f"Best weights: {after_weights.score_weights}")

Best weights: {'phonetic_score': 2, 'fuzzy_similarity_score': 3, 'entity_types_matching_preferred': 1}


## Stage 2: distance thresholds of the hierarchical likelihood

Distance from a parent beyond which a child is considered outside it, by the parent's admin level
(`PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL`). The production cascade is scaled so that the first threshold
(a region or a state) is each of the distances below, keeping the ratios between levels.

In [11]:
FIRST_DISTANCES_KM = [100, 200, 300, 400, 500]
distances = opt.optimize_parent_child_distances(evaluator, after_weights, FIRST_DISTANCES_KM)
distances.evaluations

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

,admin level 0 (km),admin level 1 (km),admin level 2 (km),admin level 3 (km),admin level 4 (km),admin level 5 (km),precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
100 / 100 / 34 / 14 / 6 / 5,100,100,34,14,6,5,0.885180,663,74,12,5,0.935780,0.992515
200 / 200 / 68 / 28 / 12 / 10,200,200,68,28,12,10,0.883845,662,76,11,5,0.935028,0.992504
300 / 300 / 102 / 42 / 18 / 15,300,300,102,42,18,15,0.882510,661,76,12,5,0.934276,0.992492
400 / 400 / 136 / 56 / 24 / 20,400,400,136,56,24,20,0.881175,660,78,11,5,0.933522,0.992481
500 / 500 / 170 / 70 / 30 / 25,500,500,170,70,30,25,0.879840,659,79,11,5,0.932767,0.992470


In [12]:
after_distances = distances.best_config
print(f"Best distances: {after_distances.parent_child_max_distance_km_by_admin_level}")

Best distances: {0: 100, 1: 100, 2: 34, 3: 14, 4: 6, 5: 5}


## Stage 3: minimum population for the population order of magnitude

Population under which `population_order_of_magnitude` is 0.

In [13]:
MIN_POPULATIONS = [0, 100, 1_000, 10_000, 100_000, 500_000, 1_000_000]
population = opt.optimize_min_population_order_of_magnitude(evaluator, after_distances, MIN_POPULATIONS)
population.evaluations

Evaluating 6 configurations:   0%|          | 0/186 [00:00<?, ?it/s]

,min_population_order_of_magnitude,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
100_000,100000,0.885180,663,74,12,5,0.935780,0.992515
500_000,500000,0.881175,660,74,15,5,0.933522,0.992481
1_000_000,1000000,0.881175,660,74,15,5,0.933522,0.992481
1_000,1000,0.881175,660,82,7,5,0.933522,0.992481
10_000,10000,0.877170,657,83,9,5,0.931254,0.992447
100,100,0.875834,656,86,7,5,0.930496,0.992436
0,0,0.874499,655,87,7,5,0.929737,0.992424


In [14]:
after_population = population.best_config
print(f"Best min population: {after_population.min_population_order_of_magnitude:_}")

Best min population: 100_000


## Stage 4: order of the factors, by a greedy search

Which factor alone decides best? With that one set, which factor decides best after it, and with which
significance threshold on the first? This repeats until all 10 factors are ordered, and a final step picks
the threshold of the last factor. The factor just added gets a threshold of 0 until the next step sets it.

The ordered factors are all primary factors: candidates no factor tells apart by its threshold are compared
again on the same factors with no threshold (see `geo_disambiguation.comparison_steps`); there are no
secondary factors.

Each factor gets 5 threshold candidates suited to the range of its address-level score (an average over the
address' entities):
- scores between 0 and 1 (`weighted_score`, `fuzzy_similarity_score`, `phonetic_score`): fine steps around the production 0.05;
- `child_parent_likelihood` and the entity type matches: coarser steps, since per-entity values are few
  (e.g. 0, 0.25, 0.75, 1) or are ratios of admin codes;
- `is_preferred_name`: 0 or 1 per entity;
- `country_likelihood_rank`: 0 to 3 per entity;
- `population_order_of_magnitude`: 0 or the rounded log10 of the population (about 2 to 7);
- `population_count`: the raw population (`DISAMBIGUATION_POPULATION_ROUNDING_FACTOR` is 1).

In [15]:
SIGNIFICANCE_THRESHOLD_CANDIDATES = {
    # in the production priority order, which breaks ties between orders
    "weighted_score": [0, 0.025, 0.05, 0.1, 0.2],
    "child_parent_likelihood": [0, 0.05, 0.1, 0.2, 0.3],
    "fuzzy_similarity_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching_preferred": [0, 0.05, 0.1, 0.2, 0.3],
    "population_order_of_magnitude": [0, 0.25, 0.5, 1, 2],
    "country_likelihood_rank": [0, 0.25, 0.5, 1, 1.5],
    "phonetic_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching": [0, 0.05, 0.1, 0.2, 0.3],
    "is_preferred_name": [0, 0.1, 0.25, 0.5, 0.75],
    "population_count": [0, 1_000, 10_000, 100_000, 1_000_000],
}

In [16]:
order_search = opt.greedy_order_search(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
order_search.summary

Evaluating 10 configurations:   0%|          | 0/310 [00:00<?, ?it/s]

Step: entity_types_matching (0)


Evaluating 45 configurations:   0%|          | 0/1395 [00:00<?, ?it/s]

Step: entity_types_matching (0.2) > weighted_score (0)


Evaluating 40 configurations:   0%|          | 0/1240 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0)


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.5)
Best order (step 11): entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.5)


,added factor,threshold set on,threshold,configurations evaluated,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
step,,,,,,,,,,,
1,entity_types_matching,NaN,NaN,10,0.669643,150,18,56,530,0.331858,0.220588
2,weighted_score,entity_types_matching,0.20,45,0.698259,361,81,75,237,0.647534,0.603679
3,population_order_of_magnitude,weighted_score,0.05,40,0.799100,533,89,45,87,0.828283,0.859677
4,phonetic_score,population_order_of_magnitude,2.00,35,0.811747,539,81,44,90,0.833720,0.856916
5,child_parent_likelihood,phonetic_score,0.10,30,0.816356,569,82,46,57,0.860166,0.908946
6,population_count,child_parent_likelihood,0.00,25,0.822819,613,126,6,9,0.896854,0.985531
7,country_likelihood_rank,population_count,100000.00,20,0.856376,638,97,10,9,0.916667,0.986090
8,fuzzy_similarity_score,country_likelihood_rank,0.00,15,0.860403,641,94,10,9,0.918996,0.986154
9,entity_types_matching_preferred,fuzzy_similarity_score,0.00,10,0.860403,641,94,10,9,0.918996,0.986154


In [17]:
opt.display_greedy_steps(order_search, top=5)

**Step 1**: factor added after nan, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching (0),entity_types_matching,None,0.669643,150,18,56,530,0.331858,0.220588
weighted_score (0),weighted_score,None,0.644269,326,68,112,248,0.603704,0.567944
entity_types_matching_preferred (0),entity_types_matching_preferred,None,0.636364,154,22,66,512,0.339207,0.231231
is_preferred_name (0),is_preferred_name,None,0.603113,155,25,77,497,0.341034,0.237730
fuzzy_similarity_score (0),fuzzy_similarity_score,None,0.597536,291,47,149,267,0.556938,0.521505


**Step 2**: factor added after entity_types_matching, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching (0.2) > weighted_score (0),weighted_score,0.20,0.698259,361,81,75,237,0.647534,0.603679
entity_types_matching (0.3) > weighted_score (0),weighted_score,0.30,0.698259,361,81,75,237,0.647534,0.603679
entity_types_matching (0) > weighted_score (0),weighted_score,0.00,0.693798,358,82,76,238,0.643885,0.600671
entity_types_matching (0.05) > weighted_score (0),weighted_score,0.05,0.693798,358,82,76,238,0.643885,0.600671
entity_types_matching (0.1) > weighted_score (0),weighted_score,0.10,0.693798,358,82,76,238,0.643885,0.600671


**Step 3**: factor added after weighted_score, and the threshold of the latter

...

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0),entity_types_matching_preferred,0.000,0.860403,641,94,10,9,0.918996,0.986154
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0.025) > entity_types_matching_preferred (0),entity_types_matching_preferred,0.025,0.860403,641,94,10,9,0.918996,0.986154
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0.05) > entity_types_matching_preferred (0),entity_types_matching_preferred,0.050,0.860403,641,94,10,9,0.918996,0.986154
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0.1) > entity_types_matching_preferred (0),entity_types_matching_preferred,0.100,0.860403,641,94,10,9,0.918996,0.986154
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0.2) > entity_types_matching_preferred (0),entity_types_matching_preferred,0.200,0.860403,641,94,10,9,0.918996,0.986154


**Step 10**: factor added after entity_types_matching_preferred, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0),is_preferred_name,0.00,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0.05) > is_preferred_name (0),is_preferred_name,0.05,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0.1) > is_preferred_name (0),is_preferred_name,0.10,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0.2) > is_preferred_name (0),is_preferred_name,0.20,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0.3) > is_preferred_name (0),is_preferred_name,0.30,0.859438,642,93,12,7,0.919771,0.989214


**Step 11**: factor added after is_preferred_name, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.5),None,0.50,0.860776,643,92,12,7,0.920544,0.989231
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.75),None,0.75,0.860776,643,92,12,7,0.920544,0.989231
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0),None,0.00,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.1),None,0.10,0.859438,642,93,12,7,0.919771,0.989214
entity_types_matching (0.2) > weighted_score (0.05) > population_order_of_magnitude (2) > phonetic_score (0.1) > child_parent_likelihood (0) > population_count (100000) > country_likelihood_rank (0) > fuzzy_similarity_score (0) > entity_types_matching_preferred (0) > is_preferred_name (0.25),None,0.25,0.859438,642,93,12,7,0.919771,0.989214


## Stage 5: significance thresholds, in the production order

Keeps the production order of the factors (primary, then secondary) and the best configuration of stage 3. Greedily,
from the first factor to the last, each factor tries the 5 threshold candidates of stage 4, with the thresholds
already chosen for the factors before it. The factor's current threshold (0.05 in production) is tried too, and
kept among ties.

In [18]:
threshold_search = opt.optimize_significance_thresholds(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
threshold_search.summary

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

weighted_score: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

child_parent_likelihood: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

is_preferred_name: 0.05 -> 0.05


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

population_count: 0.05 -> 0.05


,previous threshold,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
weighted_score,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
child_parent_likelihood,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
population_order_of_magnitude,1.00,1.00,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching_preferred,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
fuzzy_similarity_score,0.00,0.00,0.88518,663,74,12,5,0.93578,0.992515
country_likelihood_rank,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
phonetic_score,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515
is_preferred_name,0.05,0.05,0.88518,663,74,12,5,0.93578,0.992515


In [19]:
opt.display_sequential_steps(threshold_search)

**weighted_score**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score = 0.05,0.050,True,0.885180,663,74,12,5,0.935780,0.992515
weighted_score = 0.025,0.025,False,0.870321,651,84,13,6,0.926690,0.990868
weighted_score = 0,0.000,False,0.867647,649,85,14,6,0.925160,0.990840
weighted_score = 0.1,0.100,False,0.859813,644,85,20,5,0.921316,0.992296
weighted_score = 0.2,0.200,False,0.851802,638,93,18,5,0.916667,0.992224


**child_parent_likelihood**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood = 0.05,0.05,True,0.885180,663,74,12,5,0.935780,0.992515
child_parent_likelihood = 0,0.00,False,0.879840,659,78,12,5,0.932767,0.992470
child_parent_likelihood = 0.1,0.10,False,0.875834,656,81,12,5,0.930496,0.992436
child_parent_likelihood = 0.2,0.20,False,0.869159,651,85,13,5,0.926690,0.992378
child_parent_likelihood = 0.3,0.30,False,0.866489,649,87,13,5,0.925160,0.992355


**population_order_of_magnitude**

...

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching = 0.05,0.05,True,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching = 0,0.00,False,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching = 0.1,0.10,False,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching = 0.2,0.20,False,0.88518,663,74,12,5,0.93578,0.992515
entity_types_matching = 0.3,0.30,False,0.88518,663,74,12,5,0.93578,0.992515


**is_preferred_name**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name = 0.05,0.05,True,0.88518,663,74,12,5,0.935780,0.992515
is_preferred_name = 0,0.00,False,0.88518,663,74,12,5,0.935780,0.992515
is_preferred_name = 0.1,0.10,False,0.88518,663,74,12,5,0.935780,0.992515
is_preferred_name = 0.25,0.25,False,0.88518,663,74,12,5,0.935780,0.992515
is_preferred_name = 0.5,0.50,False,0.88251,661,76,12,5,0.934276,0.992492
is_preferred_name = 0.75,0.75,False,0.88251,661,76,12,5,0.934276,0.992492


**population_count**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count = 0.05,0.05,True,0.885180,663,74,12,5,0.935780,0.992515
population_count = 0,0.00,False,0.885180,663,74,12,5,0.935780,0.992515
population_count = 1000,1000.00,False,0.883845,662,74,13,5,0.935028,0.992504
population_count = 10000,10000.00,False,0.883845,662,74,13,5,0.935028,0.992504
population_count = 100000,100000.00,False,0.883845,662,74,13,5,0.935028,0.992504
population_count = 1e+06,1000000.00,False,0.883845,662,74,13,5,0.935028,0.992504


In [20]:
after_thresholds = threshold_search.best_config

## Stage 6: order, by moving factors up

Starts from the best configuration of stage 5. From the last factor of the production order to the first, each
factor moves up one position at a time while the objective is not worse than at the position before (ties keep
moving it up). It stays at the last position that was not worse. The boundary between primary and secondary
factors stays at the same position: a secondary factor moved above it becomes primary, and the last primary factor
becomes secondary.

In [21]:
order_mutation = opt.move_factors_up(evaluator, after_thresholds)
order_mutation.summary

Evaluating 9 configurations:   0%|          | 0/279 [00:00<?, ?it/s]

population_count: position 10 -> 10


Evaluating 8 configurations:   0%|          | 0/248 [00:00<?, ?it/s]

is_preferred_name: position 9 -> 9


Evaluating 7 configurations:   0%|          | 0/217 [00:00<?, ?it/s]

...

Evaluating 2 configurations:   0%|          | 0/62 [00:00<?, ?it/s]

population_order_of_magnitude: position 3 -> 3


Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

child_parent_likelihood: position 2 -> 2
weighted_score: position 1 -> 1


,from position,to position,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
population_count,10,10,0.885180,663.0,74.0,12.0,5.0,0.93578,0.992515
is_preferred_name,9,9,0.885180,663.0,74.0,12.0,5.0,0.93578,0.992515
entity_types_matching,8,8,0.885180,663.0,74.0,12.0,5.0,0.93578,0.992515
phonetic_score,7,7,0.885180,663.0,74.0,12.0,5.0,0.93578,0.992515
country_likelihood_rank,6,6,0.885180,663.0,74.0,12.0,5.0,0.93578,0.992515
fuzzy_similarity_score,5,4,0.886515,664.0,73.0,12.0,5.0,0.93653,0.992526
entity_types_matching_preferred,5,5,0.886515,664.0,73.0,12.0,5.0,0.93653,0.992526
population_order_of_magnitude,3,3,0.886515,664.0,73.0,12.0,5.0,0.93653,0.992526
child_parent_likelihood,2,2,0.886515,664.0,73.0,12.0,5.0,0.93653,0.992526


In [22]:
opt.display_sequential_steps(order_mutation)

**population_count**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count at position 9,9,True,0.877170,657,81,11,5,0.931254,0.992447
population_count at position 8,8,False,0.855808,641,100,8,5,0.918996,0.992260
population_count at position 7,7,False,0.855808,641,100,8,5,0.918996,0.992260
population_count at position 6,6,False,0.842457,631,110,8,5,0.911191,0.992138
population_count at position 5,5,False,0.830441,622,120,7,5,0.904070,0.992026
population_count at position 4,4,False,0.830441,622,120,7,5,0.904070,0.992026
population_count at position 3,3,False,0.830441,622,120,7,5,0.904070,0.992026
population_count at position 2,2,False,0.806409,604,138,7,5,0.889543,0.991790
population_count at position 1,1,False,0.583445,437,267,45,5,0.733837,0.988688


**is_preferred_name**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name at position 8,8,True,0.885180,663,74,12,5,0.935780,0.992515
is_preferred_name at position 7,7,False,0.885027,662,75,11,6,0.935028,0.991018
is_preferred_name at position 6,6,False,0.882353,660,77,11,6,0.933522,0.990991
is_preferred_name at position 5,5,False,0.862115,644,92,11,7,0.921316,0.989247
is_preferred_name at position 4,4,False,0.862115,644,92,11,7,0.921316,0.989247
is_preferred_name at position 3,3,False,0.855228,638,96,12,8,0.916667,0.987616
is_preferred_name at position 2,2,False,0.847185,632,102,12,8,0.911977,0.987500
is_preferred_name at position 1,1,False,0.822819,613,119,13,9,0.896854,0.985531


**entity_types_matching**

...

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_order_of_magnitude at position 2,2,True,0.88251,661,76,12,5,0.934276,0.992492
population_order_of_magnitude at position 1,1,False,0.82510,618,113,18,5,0.900875,0.991974


**child_parent_likelihood**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood at position 1,1,True,0.871829,653,84,12,5,0.928216,0.992401


**weighted_score**

_already first_

In [23]:
after_order_mutation = order_mutation.best_config

## Summary

In [24]:
stage_configs = {
    "production": production_config,
    "base" : base_congig,
    "1. weights": after_weights,
    "2. distances": after_distances,
    "3. min population": after_population,
    "4. greedy factor order": order_search.best_config,
    "5. thresholds": after_thresholds,
    "6. moving factors up": after_order_mutation,
}
opt.compare_stages(evaluator, stage_configs)

,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall,Δ precision,Δ correctly_linked,Δ incorrectly_linked
production,0.883845,662,75,12,5,0.935028,0.992504,0.000000,0.0,0.0
base,0.883845,662,75,12,5,0.935028,0.992504,0.000000,0.0,0.0
1. weights,0.885180,663,74,12,5,0.935780,0.992515,0.001335,1.0,-1.0
2. distances,0.885180,663,74,12,5,0.935780,0.992515,0.001335,1.0,-1.0
3. min population,0.885180,663,74,12,5,0.935780,0.992515,0.001335,1.0,-1.0
4. greedy factor order,0.860776,643,92,12,7,0.920544,0.989231,-0.023069,-19.0,17.0
5. thresholds,0.885180,663,74,12,5,0.935780,0.992515,0.001335,1.0,-1.0
6. moving factors up,0.886515,664,73,12,5,0.936530,0.992526,0.002670,2.0,-2.0


In [25]:
opt.deciding_factor_counts(evaluator, stage_configs)

,production,base,1. weights,2. distances,3. min population,4. greedy factor order,5. thresholds,6. moving factors up
(common parent of tied candidates),8,8,9,9,9,10,9,9
(single candidate),91,91,91,91,91,91,91,91
child_parent_likelihood,68,68,71,71,71,47,71,71
country_likelihood_rank,24,24,24,24,24,30,24,24
entity_types_matching,26,26,26,26,26,45,26,26
entity_types_matching (no threshold),0,0,0,0,0,1,0,0
entity_types_matching_preferred,1,1,1,1,1,0,1,0
fuzzy_similarity_score,35,35,24,24,24,25,24,25
is_preferred_name,9,9,8,8,8,13,8,8
phonetic_score,5,5,4,4,4,5,4,4


In [ ]:
import datetime
print(datetime.datetime.now())
print(opt.describe_config(after_order_mutation))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'population_order_of_magnitude',
    'fuzzy_similarity_score',
    'entity_types_matching_preferred',
    'country_likelihood_rank',
    'phonetic_score',
    'entity_types_matching',
    'is_preferred_name',
    'population_count',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'phonetic_score': 2,
    'fuzzy_similarity_score': 3,
    'entity_types_matching_preferred': 1,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 100,
    1: 100,
    2: 34,
    3: 14,
    4: 6,
    5: 5,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 100_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'population_order_of_magnitude': 1,
    'entity_types_matching_preferred': 0.05,
    'fuzzy_similarity_score

In [27]:
evaluator.close()